# Tutorial 19: dCDH for Marketing Pulse Campaigns

A practitioner walkthrough for measuring lift from promotional campaigns that turn on AND off across markets at staggered times. The tutorial uses the `ChaisemartinDHaultfoeuille` estimator (alias `DCDH`) - diff-diff's only estimator built for reversible (non-absorbing) treatment, where every other modern staggered estimator in the library assumes treatment is absorbing.

## 1. The Marketing Pulse Problem

Your team runs paid-promo pulses across 60 markets. Some markets ran the promo at the start of the quarter and turned it off as the campaign budget rolled to the next geo; others started untreated and switched the promo on at some point during the quarter. Leadership wants the average lift on weekly checkout sessions while the promo was on.

**Why this is hard.** Three things break standard methods:

1. **Treatment is reversible.** This panel has both joiners (markets that switched the promo on) and leavers (markets that switched it off). The canonical staggered-DiD estimators - Callaway-Sant'Anna, Sun-Abraham, Wooldridge ETWFE, ImputationDiD - all assume *absorbing* treatment: once treated, always treated. They simply don't apply when the promo can come back off.

2. **Two-way fixed-effects regression silently uses negative weights.** When you have switchers in both directions in the same panel, OLS with unit and time fixed effects ends up using some treated cells as *controls* for other treated cells, weighting those cells negatively. Under heterogeneous treatment effects, those negative weights can attenuate or even flip the sign of the regression coefficient ([de Chaisemartin & D'Haultfoeuille 2020](https://www.aeaweb.org/articles?id=10.1257/aer.20181169), Theorem 1).

3. **No diagnostic tells you when to worry.** The standard error from the OLS regression doesn't reveal the weighting problem. You need a separate decomposition to know whether to trust the regression coefficient or reach for an alternative.

**Why diff-diff.** The library implements `ChaisemartinDHaultfoeuille` (`DCDH`) following the AER 2020 paper plus its [dynamic companion](https://www.nber.org/papers/w29873). Phase 1 ships the contemporaneous-switch estimator `DID_M` plus a joiners-vs-leavers decomposition; the multi-horizon event study via `L_max` adds dynamic effects with multiplier-bootstrap inference. Critically, the library also exposes the AER 2020 Theorem 1 TWFE decomposition as a standalone diagnostic - so you can quantify how badly TWFE is contaminated *before* you reach for the fix. Implementation details and any documented deviations from R's `did_multiplegt_dyn` reference live in [`docs/methodology/REGISTRY.md`](../methodology/REGISTRY.html).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from diff_diff import (
    DCDH,
    generate_reversible_did_data,
    twowayfeweights,
)

plt.style.use("seaborn-v0_8-whitegrid")

## 2. The Data

We'll simulate a panel that mirrors a marketing pulse campaign:

- **60 markets**, each observed for **8 weeks**
- Some markets started the quarter with the promo on and switched it off (leavers); others started untreated and switched the promo on (joiners). Each market switches exactly once during the panel - the [A5 single-switch contract](../methodology/REGISTRY.html) the analytical SE is derived under.
- Outcome: weekly checkout sessions per market, baseline ~110
- True treatment effect: **+12 sessions per market-week** when the promo is on, with cell-level effect heterogeneity (some markets respond more strongly than others).

We use `generate_reversible_did_data` with `pattern="single_switch"` and `heterogeneous_effects=True`. Because the data is synthetic, the true effect is known and we can verify dCDH recovers it.

In [ ]:
raw = generate_reversible_did_data(
    n_groups=60,
    n_periods=8,
    pattern="single_switch",
    initial_treat_frac=0.4,
    treatment_effect=12.0,
    heterogeneous_effects=True,
    effect_sd=4.0,
    group_fe_sd=8.0,
    time_trend=0.5,
    noise_sd=2.0,
    seed=53,  # locked via seed-search; see _scratch/dcdh_tutorial/
)
df = raw.rename(
    columns={
        "group": "market_id",
        "period": "week",
        "treatment": "promo_on",
        "outcome": "sessions",
    }
)
df["sessions"] = df["sessions"] + 100.0  # shift to a realistic baseline

print(f"Panel shape: {df.shape}")
print(f"Markets: {df['market_id'].nunique()}")
print(f"Weeks: {sorted(df['week'].unique())}")
print(f"Sessions range: [{df['sessions'].min():.0f}, {df['sessions'].max():.0f}]")

In [ ]:
# Switcher-type counts. With pattern="single_switch" every group
# switches exactly once, so we have only joiners (0 → 1) and leavers
# (1 → 0); no never-treated or always-treated groups by construction.
df.groupby("switcher_type").size()

In [ ]:
# Mean sessions over time, split by which direction the market switched.
first_treat = df.groupby("market_id")["promo_on"].first()
category = df["market_id"].map(
    lambda m: "starts off, switches on" if first_treat[m] == 0 else "starts on, switches off"
)
df_plot = df.assign(category=category)

fig, ax = plt.subplots(figsize=(9, 5))
for label, color in [("starts off, switches on", "#1f77b4"), ("starts on, switches off", "#d62728")]:
    weekly = df_plot[df_plot["category"] == label].groupby("week")["sessions"].mean()
    ax.plot(weekly.index, weekly.values, label=label, color=color, marker="o", linewidth=2)
ax.set_xlabel("Week")
ax.set_ylabel("Mean weekly sessions")
ax.set_title("Marketing pulses on/off across markets — outcomes by switcher type")
ax.legend(loc="upper left")
plt.show()

## 3. Why Standard Regression Misleads Here

Before reaching for dCDH, fit standard two-way fixed effects (TWFE) regression on this panel and read out the diagnostic. The dCDH authors derived a closed-form decomposition of the TWFE coefficient (Theorem 1, AER 2020) that tells you *quantitatively* how badly the regression is contaminated, before you have to trust any alternative estimator.

The library exposes this as a standalone function, `twowayfeweights`, that returns three numbers:

- `beta_fe`: the plain TWFE coefficient on the treatment indicator.
- `fraction_negative`: the share of treated cells that receive a *negative* weight in the TWFE coefficient. Any positive value is a warning sign - it means OLS is using some treated units as controls for other treated units.
- `sigma_fe`: the smallest cell-level effect-heterogeneity standard deviation that could flip the sign of the TWFE coefficient. Small `sigma_fe` (relative to plausible heterogeneity in your domain) means the regression is fragile.

In [ ]:
twfe = twowayfeweights(
    df,
    outcome="sessions",
    group="market_id",
    time="week",
    treatment="promo_on",
)

print(f"TWFE coefficient (beta_fe):       {twfe.beta_fe:.3f}")
print(f"Fraction of negative weights:     {twfe.fraction_negative:.3f}  ({twfe.fraction_negative*100:.1f}%)")
print(f"Sign-flip threshold (sigma_fe):   {twfe.sigma_fe:.3f}")
print(f"True treatment effect (DGP):      12.000")

**Plain-English interpretation.** The TWFE regression estimates the lift at about **11.5 sessions per market-week** - close to the true effect of 12.0 in this synthetic panel. But the diagnostic surfaces two warning signs: **15.4% of treated cells receive negative weight** in that estimate, and the sign-flip threshold sigma_fe is about 12.3. In domains where you might plausibly believe cell-level treatment effects vary by ~12 sessions in standard deviation, the TWFE coefficient is fragile.

On *this* panel the bias happens to be modest because effect heterogeneity is moderate. In production data with stronger heterogeneity the bias would grow significantly. The point of the diagnostic isn't to tell you that TWFE is *catastrophically* wrong today - it's to tell you that TWFE *could* swing on data you haven't seen yet, and to surface the structural problem before you trust the regression coefficient.

In [ ]:
# Top 15 cells with the most-negative TWFE weights, colored red.
weights = twfe.weights.sort_values("weight").head(15)
labels = [
    f"M{int(r.market_id)}, wk{int(r.week)}"
    for r in weights.itertuples()
]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#d62728" if w < 0 else "#1f77b4" for w in weights["weight"]]
ax.barh(range(len(weights)), weights["weight"].values, color=colors)
ax.set_yticks(range(len(weights)))
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color="black", linewidth=0.7)
ax.set_xlabel("TWFE weight on this cell")
ax.set_title(
    f"Top 15 cells with most-negative TWFE weights\n"
    f"({twfe.fraction_negative*100:.1f}% of all {len(twfe.weights)} cells receive negative weight)"
)
plt.show()

**The transition.** We need an estimator that only compares each switching cell to *contemporaneously stable* control cells - never to other switchers. That's what `DID_M` from the dCDH framework does, by construction.

## 4. dCDH Phase 1: DID_M, Joiners, Leavers, Placebo

`DID_M` is the average across periods of two pieces:

- **DID_+** (joiners): markets switching `0 → 1` between consecutive periods, compared to *contemporaneously untreated* control cells.
- **DID_-** (leavers): markets switching `1 → 0`, compared to *contemporaneously treated* control cells.

Both pieces use only cells whose treatment status was stable across the two periods being compared - so no treated unit is ever used as a control for another treated unit. The library reports DID_+, DID_-, and their average DID_M separately, so you can see if the two halves agree.

**Where do the controls come from?** dCDH's controls are *contemporaneously stable cells*, not a permanently-untreated comparison group. A market that's untreated at week 3 and week 4 contributes a stable-untreated cell at week 4 - even if that same market eventually turns the promo on at week 5 and keeps it on through week 8. Symmetrically, a market that's been running the promo since week 1 and is still running it at week 4 contributes a stable-treated cell at week 4. This is what lets dCDH work on panels with **no permanent never-treated markets at all** - our panel has zero never-treated and zero always-treated units, only 60 switchers. Among diff-diff's modern staggered-DiD estimators - Callaway-Sant'Anna, Sun-Abraham, Wooldridge ETWFE, ImputationDiD, TwoStageDiD, EfficientDiD - all assume absorbing treatment, so the question of which controls they use only arises in panels where treatment never switches off. dCDH applies in the broader reversible-treatment setting and uses contemporaneous stability rather than a permanent never-treated cohort. The technical condition - de Chaisemartin & D'Haultfoeuille's Assumption 11 - is that at every period when a switcher exists, at least one stable cell of the relevant type also exists. The check is **per-period**, not on whole-panel totals: 154 stable-untreated cells aggregated across the panel doesn't prove anything if some specific switching week happened to have none. The library checks A11 at fit time period-by-period and emits a `UserWarning` (zeroing the offending period's contribution by paper convention) if any switching period lacks stable controls. Our fit above ran without such a warning, so A11 holds at every switching week in this DGP. Single-switch panels also tend to satisfy A11 by construction because each cohort's pre-switch and post-switch periods naturally function as stable cells for cohorts that switch at adjacent times.

The library also computes a **single-lag placebo** `DID_M^pl`: the same DID_M machinery shifted one period back. Under parallel pre-trends the placebo should be near zero. (Note: Phase 1's single-lag placebo SE is `NaN` by design - the per-period aggregation path doesn't have an analytical influence-function derivation. Magnitude-only interpretation here; full inference comes from the multi-horizon placebos in Section 5 below.)

In [ ]:
model = DCDH(twfe_diagnostic=True, placebo=True, seed=42)
results = model.fit(
    df,
    outcome="sessions",
    group="market_id",
    time="week",
    treatment="promo_on",
)
print(results.summary())

**Plain-English interpretation.** dCDH estimates the headline lift at **about 11.2 sessions per market-week** (95% CI: ~10.1 to 12.3), covering the true effect of 12.0. The TWFE coefficient was 11.5 - the two estimators happen to land close on this panel because effect heterogeneity is modest, but **dCDH guarantees no negative-weight contamination by construction**, while TWFE only happened to escape it this time.

The TWFE diagnostic block at the bottom of the summary repeats the numbers from Section 3 (15.4% negative weights, sigma_fe ≈ 12.3) as a built-in cross-check - the library computes the diagnostic automatically when `twfe_diagnostic=True` (the default).

In [ ]:
# Joiners vs leavers breakdown
jl = results.to_dataframe(level="joiners_leavers")
jl.round(3)

**Reading joiners vs leavers.** Both halves should produce a positive lift in a healthy marketing-pulse design - turning the promo on increases sessions, and turning it off decreases them. Here DID_+ ≈ 11.0 and DID_- ≈ 11.9: both substantially positive, both within sampling uncertainty of each other and of the true effect of 12. If they had disagreed by sign or by a large margin (say one was 5 and the other was 20), that would be a heterogeneity signal worth investigating before reporting one number to leadership.

In [ ]:
# Placebo magnitude check (SE is NaN for Phase 1 single-lag)
print(f"Placebo effect:      {results.placebo_effect:.3f}")
print(f"|Placebo / DID_M|:   {abs(results.placebo_effect / results.overall_att):.2%}")
print()
print("Placebo magnitude is small (~8% of DID_M), supporting parallel")
print("pre-trends. Full placebo inference with bootstrap CIs comes from")
print("the multi-horizon event study below.")

## 5. Multi-Horizon Event Study with Bootstrap

DID_M collapses the dynamic effect to one number - the average lift across all switching cells. Setting `L_max=L` instead computes `DID_l` for each horizon `l = 1..L` after each switch, plus `DID^pl_l` placebos at horizons `l = -L..-1`. This tells you whether the on-impact lift is sustained or fades, and whether the pre-treatment placebos sit on zero.

With `L_max=2` we get two post-switch horizons and two placebo horizons. The multiplier bootstrap (`n_bootstrap=199`, matching the library's `ci_params.bootstrap` convention) gives valid CIs at every horizon, including the placebo horizons.

In [ ]:
# Narrow filter: silence the EXPECTED Assumption 7 warning (cost-benefit
# delta is computed on the full sample when leavers are present), and
# let any new / unexpected UserWarning surface so the notebook stays
# usable as a drift detector.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"Assumption 7 .* is violated: leavers present",
        category=UserWarning,
    )
    model_es = DCDH(
        twfe_diagnostic=False, placebo=True, n_bootstrap=199, seed=42
    )
    results_es = model_es.fit(
        df,
        outcome="sessions",
        group="market_id",
        time="week",
        treatment="promo_on",
        L_max=2,
    )

es_df = results_es.to_dataframe(level="event_study")
es_df.round(3)

In [ ]:
# Event-study errorbar plot with bootstrap CIs.
es_plot = es_df[es_df["horizon"] != 0]  # drop reference row
is_pre = es_plot["horizon"] < 0

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(
    es_plot.loc[is_pre, "horizon"],
    es_plot.loc[is_pre, "effect"],
    yerr=[
        es_plot.loc[is_pre, "effect"] - es_plot.loc[is_pre, "conf_int_lower"],
        es_plot.loc[is_pre, "conf_int_upper"] - es_plot.loc[is_pre, "effect"],
    ],
    fmt="o", color="#888888", capsize=4, label="Pre-treatment placebos",
)
ax.errorbar(
    es_plot.loc[~is_pre, "horizon"],
    es_plot.loc[~is_pre, "effect"],
    yerr=[
        es_plot.loc[~is_pre, "effect"] - es_plot.loc[~is_pre, "conf_int_lower"],
        es_plot.loc[~is_pre, "conf_int_upper"] - es_plot.loc[~is_pre, "effect"],
    ],
    fmt="o", color="#1f77b4", capsize=4, label="Post-treatment effects",
)
ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
ax.axhline(12.0, color="green", linewidth=0.8, linestyle=":", label="true effect = 12.0")
ax.set_xlabel("Weeks since promo switched")
ax.set_ylabel("Effect on weekly sessions")
ax.set_title("dCDH event study (L_max=2, multiplier bootstrap)")
ax.legend(loc="lower right")
plt.show()

**Reading the event study.**

- **Both placebo horizons** (l = -2 and l = -1) sit on zero with confidence intervals comfortably covering it. Pre-trends look parallel - we have no evidence that something other than the promo was driving session growth in the cells we're using as controls.
- **On-impact effect** at l = 1 is about **+11.2 sessions** with a 95% bootstrap CI of roughly [9.7, 12.8], covering the true effect of 12.
- **Sustained effect** at l = 2 is **+11.3 sessions** with CI [10.0, 12.6]. The lift didn't fade in the second week post-switch.

Bootstrap CIs reflect the cohort-recentered influence-function variance with the finite-sample stability the multiplier bootstrap provides. The fact that both horizons agree closely with each other AND with the headline `DID_M` from Section 4 (the per-period and per-group aggregation paths converge) is a built-in consistency check.

## 6. Communicating Results to Leadership

A stakeholder-ready summary of the synthetic walkthrough above. Each bullet pulls from a specific section of the analysis:

> **Headline.** The pulse campaign lifted weekly checkout sessions by approximately **11.2 sessions per market per week** while the promo was on (95% CI: 10.1 to 12.3). On a baseline of about 110 weekly sessions per market, that's roughly a **10% lift**. *[Source: `results.overall_att` from Section 4.]*
>
> **Sample size and design.** 60 markets observed for 8 weeks (480 market-weeks). Of those, 43 markets started untreated and switched the promo on at some point during the quarter (joiners), and 17 markets started with the promo on and switched it off (leavers). Method: dCDH (de Chaisemartin & D'Haultfoeuille 2020) - diff-diff's only estimator built for treatment that can switch on AND off in the same panel. *[Source: switcher_type counts and panel shape from Section 2.]*
>
> **Validity evidence.** Three checks supported the result. (a) The TWFE diagnostic flagged 15.4% of cells with negative weight in the standard regression, signaling that we needed an alternative - dCDH avoids that contamination by construction. (b) The single-lag placebo from the per-period aggregation was small (~0.9 sessions, ~8% of the headline). (c) The multi-horizon placebos at l = -2 and l = -1 both sat on zero with bootstrap CIs comfortably covering it - parallel pre-trends look credible. *[Sources: TWFE diagnostic from Section 3, single-lag placebo from Section 4, multi-horizon placebos from Section 5.]*
>
> **What "+11.2 sessions per market per week" means in business terms.** Across 60 markets and the weeks each one had the promo on, that's the per-market-week lift attributable to the campaign. Translate to your own revenue-per-session to compare against campaign spend, then use the per-market lift estimate to project what scaling the promo to additional markets would deliver. *[Source: business framing of the headline.]*
>
> **Practical significance caveat.** The 10% lift is statistically significant (bootstrap p < 0.01 at both post-treatment horizons), and the on-impact effect persists at the second horizon - the pulse worked while it was on. Whether 10% justifies the campaign cost is a business judgment, not a statistical one. Note also that joiners (DID_+ ≈ 11.0) and leavers (DID_- ≈ 11.9) gave consistent signals, which reduces the worry that the average is hiding heterogeneity between starting and stopping the promo. *[Sources: dynamic horizons from Section 5, joiners/leavers breakdown from Section 4.]*

Adapt this template for your own campaign by swapping in your numbers from `results.summary()`, your own market and switcher counts, your own validity diagnostics, and your own business translation. The pattern - **headline → sample size and design → validity evidence → business interpretation → practical significance** - is the part to keep.

In [ ]:
# Drift guards: tolerance-based asserts that lock the numbers quoted in
# the Section 4 / Section 5 narrative and the Section 6 stakeholder
# template. nbmake will fail if generate_reversible_did_data() or DCDH
# output drifts outside these ranges, forcing the markdown to be
# updated before this notebook can pass CI.
#
# Asserts pull from BOTH `results` (Section 4 single-horizon fit) AND
# `results_es` (Section 5 multi-horizon fit) - both fits are still
# in scope above this cell.

# Section 4 (L_max=None): per-period DID_M path
assert 10.72 <= results.overall_att <= 11.72, results.overall_att
assert results.overall_conf_int[0] <= 12.0 <= results.overall_conf_int[1]
assert abs(results.placebo_effect) < 1.5, results.placebo_effect
assert results.twfe_fraction_negative >= 0.10  # documents TWFE bias signal

# Section 5 (L_max=2): per-group DID_g,1 path - DIFFERENT compute path
# than overall_att, NOT bit-identical. Verified in seed-search to agree
# on truth-coverage at seed=53.
_h1 = results_es.event_study_effects[1]["effect"]
assert 10.24 <= _h1 <= 12.24, _h1
assert (
    results_es.event_study_effects[1]["conf_int"][0]
    <= 12.0
    <= results_es.event_study_effects[1]["conf_int"][1]
)

print("All drift guards passed.")

## 7. Extensions and Where to Go Next

This tutorial covered the dCDH **Phase 1** surface (DID_M, joiners/leavers decomposition, single-lag placebo, TWFE diagnostic) plus the **multi-horizon event study with bootstrap** (`L_max`, `n_bootstrap`). The library also supports several extensions that we did not demonstrate here:

- **Per-trajectory disaggregation** (`by_path=k`): when joiners and leavers each follow a few common treatment paths (e.g., on-off-on vs on-on-off), `by_path=k` reports the event study separately for the top-k most common observed paths. Useful for pulse campaigns where the schedule varies across markets.
- **Group-specific linear trends** (`trends_linear=True`): allows each market to have its own pre-treatment slope, absorbing differential trends.
- **State-set-specific trends** (`trends_nonparam=...`): allows non-parametric trends shared within state-set strata.
- **HonestDiD sensitivity analysis** (`honest_did=True`): Rambachan-Roth (2023) bounds on the post-treatment effects under controlled parallel-trends violations, computed on the placebo event-study surface.
- **Survey-design support** (`survey_design=...`): Taylor-series linearization with sampling weights, strata, PSU, and FPC.

See [`docs/api/chaisemartin_dhaultfoeuille.rst`](../api/chaisemartin_dhaultfoeuille.html) for the full parameter reference and [`docs/methodology/REGISTRY.md`](../methodology/REGISTRY.html) for the methodology contract on each surface.

**Related tutorials.**

- [Tutorial 1: Basic DiD](01_basic_did.ipynb) - the 2x2 building block dCDH generalizes.
- [Tutorial 2: Staggered DiD](02_staggered_did.ipynb) - Goodman-Bacon decomposition is the staggered-adoption analog of the TWFE diagnostic shown here.
- [Tutorial 5: HonestDiD](05_honest_did.ipynb) - sensitivity to parallel-trends violations on event studies; works on dCDH's placebo surface via `honest_did=True`.
- [Tutorial 17: Brand Awareness Survey](17_brand_awareness_survey.ipynb) - reach for this if you have survey data with sampling weights / strata / PSU instead of a panel.
- [Tutorial 18: Geo-Experiment Analysis](18_geo_experiments.ipynb) - reach for this if you have a single-launch pilot in a small number of test markets.

**Summary: when to reach for dCDH.**

1. Use dCDH when treatment is **reversible** - the panel has switchers in both directions (joiners and leavers) in the same data. No other modern Python DiD estimator handles this case.
2. Run `twowayfeweights` *before* fitting any estimator on a reversible panel - the diagnostic tells you whether to worry about TWFE contamination, in numbers (`fraction_negative`, `sigma_fe`).
3. Read joiners (`DID_+`) and leavers (`DID_-`) separately. Disagreement between the two halves is heterogeneity worth investigating before averaging into one number for stakeholders.
4. Use `L_max` + multiplier bootstrap to expose the dynamic structure of the effect - is the lift on-impact only, sustained, or fading? - and to get valid placebo CIs that the Phase 1 single-lag placebo can't provide.
5. Defer to follow-up tutorials for `by_path`, `trends_linear`/`trends_nonparam`, HonestDiD on dCDH's placebo surface, and the survey-design integration. Each is a single constructor or `fit()` kwarg away.